In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata


# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50, data=None):
    if data is None:
        data = qndata.stocks.load_data()

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns matrix
# ============================================================
def compute_returns_matrix(data, chosen_assets, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


# ============================================================
# 3) Covariance estimators: SampleCov and EWMA
# ============================================================
def make_psd_matrix_np(Sigma, ridge=1e-6, eps=1e-10):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    vals, vecs = np.linalg.eigh(S)
    vals = np.maximum(vals, eps)
    return vecs @ np.diag(vals) @ vecs.T


def compute_sigma_series_sample(R, lookback=252, annualize=252, shrink_delta=0.0, ridge=1e-6):
    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback + 1}.")

    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t + 1, :]

        if window.shape[0] > 2:
            Sigma = np.cov(window, rowvar=False)
            Sigma = 0.5 * (Sigma + Sigma.T)

            if shrink_delta > 0.0:
                D = np.diag(np.diag(Sigma))
                Sigma = (1.0 - shrink_delta) * Sigma + shrink_delta * D
        else:
            Sigma = np.eye(n) * 1e-6

        Sigma_series[t] = make_psd_matrix_np(Sigma * annualize, ridge=ridge)

    return Sigma_series


def compute_sigma_series_ewma(R, lookback_init=252, annualize=252, alpha=0.90, ridge=1e-6):
    T, n = R.shape
    if T <= lookback_init + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback_init + 1}.")

    Sigma_series = np.zeros((T, n, n))

    init_end = min(T, lookback_init + 1)
    init_win = R[:init_end, :]

    if init_win.shape[0] > 2:
        S = np.cov(init_win, rowvar=False)
    else:
        S = np.eye(n) * 1e-6

    S = make_psd_matrix_np(S * annualize, ridge=ridge)

    for t in range(T):
        if t > 0:
            r_t = R[t].reshape(-1, 1)
            S = alpha * S + (1.0 - alpha) * annualize * (r_t @ r_t.T)
            S = make_psd_matrix_np(S, ridge=ridge)

        Sigma_series[t] = S.copy()

    return Sigma_series


# ============================================================
# 4) Alpha models: simple vs AR(1)
# ============================================================
def mu_series_simple(R, lookback_mu=60, mode="momentum", annualize=252):
    T, n = R.shape
    mu = np.zeros((T, n))

    for t in range(T):
        start = max(0, t - lookback_mu + 1)
        window = R[start:t + 1, :]
        m = np.nanmean(window, axis=0)

        if mode == "mean_reversion":
            m = -m

        mu[t] = m * annualize

    return mu


def mu_series_ar1(R, lookback_model=252, annualize=252, smooth_alpha=0.20):
    T, n = R.shape
    mu_raw = np.zeros((T, n))

    for t in range(T):
        start = max(1, t - lookback_model + 1)
        window = R[start:t + 1, :]

        if window.shape[0] < 3:
            mu_raw[t] = 0.0
            continue

        X = window[:-1, :]
        Y = window[1:, :]

        for i in range(n):
            x = X[:, i]
            y = Y[:, i]
            A = np.column_stack([np.ones_like(x), x])

            try:
                theta, *_ = np.linalg.lstsq(A, y, rcond=None)
                a, b = theta
                mu_raw[t, i] = (a + b * R[t, i]) * annualize
            except Exception:
                mu_raw[t, i] = 0.0

    if smooth_alpha is None or smooth_alpha <= 0.0:
        return mu_raw

    mu_sm = np.zeros_like(mu_raw)
    mu_sm[0] = mu_raw[0]

    for t in range(1, T):
        mu_sm[t] = smooth_alpha * mu_raw[t] + (1.0 - smooth_alpha) * mu_sm[t - 1]

    return mu_sm


# ============================================================
# 5) Time-varying transaction cost series
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    T, _ = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    if mode == "vol_scaled":
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t + 1])

        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0

        vol_state_norm = vol_state / denom
        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 6) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-6, eps=1e-10):
    return make_psd_matrix_np(Sigma, ridge=ridge, eps=eps)


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            s2 = w.sum()
            if s2 <= 0:
                w = np.ones_like(w) / len(w)
            else:
                w = w / s2
    else:
        w = w / s

    return w


def turnover_from_weights(W):
    if W.shape[0] < 2:
        return np.array([])
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    if len(rp) == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "cum_return": np.nan,
        }

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1) if len(rp) > 1 else 0.0
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 7) Basic + One-Step
# ============================================================
def solve_basic_one_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-6
):
    n = len(mu)
    w = cp.Variable(n)
    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
    )

    constraints = add_constraints(w, long_only=True, w_max=w_max, fully_invested=True)
    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


# ============================================================
# 8) Basic + Two-Step
# ============================================================
def solve_basic_two_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    rho=25.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-6
):
    n = len(mu)
    S = make_psd_matrix(Sigma, ridge=ridge)

    w_star = cp.Variable(n)
    objective1 = cp.Maximize(
        mu @ w_star - gamma * cp.quad_form(w_star, S)
    )
    constraints1 = add_constraints(w_star, long_only=True, w_max=w_max, fully_invested=True)
    prob1 = cp.Problem(objective1, constraints1)
    ok1 = solve_problem(prob1)

    if not ok1:
        w_star_val = normalise_weights(w_prev)
    else:
        w_star_val = normalise_weights(w_star.value, fallback=w_prev)

    w = cp.Variable(n)
    trade = w - w_prev

    objective2 = cp.Minimize(
        (rho / 2.0) * cp.sum_squares(w - w_star_val)
        + lam_t * cp.norm1(trade)
    )
    constraints2 = add_constraints(w, long_only=True, w_max=w_max, fully_invested=True)
    prob2 = cp.Problem(objective2, constraints2)
    ok2 = solve_problem(prob2)

    if not ok2:
        w_val = normalise_weights(w_prev)
    else:
        w_val = normalise_weights(w.value, fallback=w_prev)

    return w_star_val, w_val


# ============================================================
# 9) Enhanced + One-Step
# ============================================================
def solve_enhanced_one_step(
    mu,
    Sigma,
    w_prev,
    w_bench,
    gamma=5.0,
    eta=2.0,
    lam_t=0.0020,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-6
):
    n = len(mu)
    w = cp.Variable(n)
    S = make_psd_matrix(Sigma, ridge=ridge)

    trade = w - w_prev
    abs_risk = cp.quad_form(w, S)
    te_risk = cp.quad_form(w - w_bench, S)

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-10))
    D = np.diag(vol)
    robust_penalty = cp.norm2(D @ w)

    objective = cp.Maximize(
        mu @ w
        - gamma * abs_risk
        - eta * te_risk
        - lam_t * cp.norm1(trade)
        - kappa * robust_penalty
        - 1.0 * cp.sum_squares(trade)
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


# ============================================================
# 10) Enhanced + Two-Step
# ============================================================
def solve_enhanced_two_step(
    mu,
    Sigma,
    w_prev,
    w_bench,
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    lam_t=0.0020,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-6
):
    n = len(mu)
    S = make_psd_matrix(Sigma, ridge=ridge)

    w_star = cp.Variable(n)

    abs_risk_star = cp.quad_form(w_star, S)
    te_risk_star = cp.quad_form(w_star - w_bench, S)

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-10))
    D = np.diag(vol)
    robust_penalty_star = cp.norm2(D @ w_star)

    objective1 = cp.Maximize(
        mu @ w_star
        - gamma * abs_risk_star
        - eta * te_risk_star
        - kappa * robust_penalty_star
    )
    constraints1 = add_constraints(w_star, long_only=True, w_max=w_max, fully_invested=True)
    prob1 = cp.Problem(objective1, constraints1)
    ok1 = solve_problem(prob1)

    if not ok1:
        w_star_val = normalise_weights(w_prev)
    else:
        w_star_val = normalise_weights(w_star.value, fallback=w_prev)

    w = cp.Variable(n)
    trade = w - w_prev

    objective2 = cp.Minimize(
        (rho / 2.0) * cp.sum_squares(w - w_star_val)
        + lam_t * cp.norm1(trade)
        + 1.0 * cp.sum_squares(trade)
    )
    constraints2 = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob2 = cp.Problem(objective2, constraints2)
    ok2 = solve_problem(prob2)

    if not ok2:
        w_val = normalise_weights(w_prev)
    else:
        w_val = normalise_weights(w.value, fallback=w_prev)

    return w_star_val, w_val


# ============================================================
# 11) Run one combination through time
# ============================================================
def run_strategy_combo(
    mu_series,
    Sigma_series,
    cost_series,
    family="basic",
    method="one",
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-6,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    w_bench = np.ones(n) / n
    W = np.zeros((T, n))
    W_star = np.full((T, n), np.nan)

    for t in range(T):
        mu_t = mu_series[t]
        Sigma_t = Sigma_series[t]
        lam_t = cost_series[t]

        if family == "basic" and method == "one":
            w_t = solve_basic_one_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                lam_t=lam_t,
                w_max=w_max,
                ridge=ridge
            )

        elif family == "basic" and method == "two":
            w_star_t, w_t = solve_basic_two_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                rho=rho,
                lam_t=lam_t,
                w_max=w_max,
                ridge=ridge
            )
            W_star[t] = w_star_t

        elif family == "enhanced" and method == "one":
            w_t = solve_enhanced_one_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                w_bench=w_bench,
                gamma=gamma,
                eta=eta,
                lam_t=lam_t,
                kappa=kappa,
                tau=tau,
                w_max=w_max,
                ridge=ridge
            )

        elif family == "enhanced" and method == "two":
            w_star_t, w_t = solve_enhanced_two_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                w_bench=w_bench,
                gamma=gamma,
                eta=eta,
                rho=rho,
                lam_t=lam_t,
                kappa=kappa,
                tau=tau,
                w_max=w_max,
                ridge=ridge
            )
            W_star[t] = w_star_t

        else:
            raise ValueError("Invalid family/method combination.")

        W[t] = w_t
        w_prev = w_t

    return W, W_star


# ============================================================
# 12) Realised returns
# ============================================================
def realised_returns_gross(R, W):
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    gross = realised_returns_gross(R, W)
    T = W.shape[0]

    trade_costs = np.zeros(T - 1)
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 13) Summarise one strategy
# ============================================================
def summarise_strategy(name, W, W_star, R_daily, cost_series):
    to = turnover_from_weights(W)
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    return {
        "name": name,
        "W": W,
        "W_star": W_star,
        "turnover": to,
        "rp_gross": rp_gross,
        "rp_net": rp_net,
        "tc_paid": tc_paid,
        "eq_gross": eq_gross,
        "eq_net": eq_net,
        "stats_gross": stats_gross,
        "stats_net": stats_net,
        "avg_turnover": np.mean(to) if len(to) > 0 else np.nan,
        "max_turnover": np.max(to) if len(to) > 0 else np.nan,
        "avg_max_weight": float(np.mean(np.max(W, axis=1))) if W.size > 0 else np.nan,
        "has_nan_weights": bool(np.isnan(W).any()),
        "avg_cost_coeff": float(np.mean(cost_series)),
        "avg_realised_tc": float(np.mean(tc_paid)) if len(tc_paid) > 0 else np.nan,
    }


# ============================================================
# 14) CSV save helpers
# ============================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_chosen_assets_csv(seed_dir, seed, chosen_assets):
    df = pd.DataFrame({
        "seed": seed,
        "asset": chosen_assets
    })
    df.to_csv(os.path.join(seed_dir, f"chosen_assets_seed_{seed}.csv"), index=False)
    return df


def save_summary_csv(seed_dir, seed, results):
    rows = []
    for r in results:
        rows.append({
            "seed": seed,
            "strategy": r["name"],
            "final_equity_gross": r["eq_gross"][-1] if len(r["eq_gross"]) > 0 else np.nan,
            "final_equity_net": r["eq_net"][-1] if len(r["eq_net"]) > 0 else np.nan,
            "cum_return_gross": r["stats_gross"]["cum_return"],
            "cum_return_net": r["stats_net"]["cum_return"],
            "sharpe_gross": r["stats_gross"]["sharpe"],
            "sharpe_net": r["stats_net"]["sharpe"],
            "max_drawdown_gross": r["stats_gross"]["max_drawdown"],
            "max_drawdown_net": r["stats_net"]["max_drawdown"],
            "avg_turnover": r["avg_turnover"],
            "max_turnover": r["max_turnover"],
            "avg_max_weight": r["avg_max_weight"],
            "avg_cost_coeff": r["avg_cost_coeff"],
            "avg_realised_tc": r["avg_realised_tc"],
            "has_nan_weights": r["has_nan_weights"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"summary_seed_{seed}.csv"), index=False)
    return df


def save_timeseries_csv(seed_dir, seed, times, results):
    rows = []
    time_index = pd.to_datetime(times[1:])

    for r in results:
        for i, dt in enumerate(time_index):
            rows.append({
                "seed": seed,
                "date": dt,
                "strategy": r["name"],
                "eq_gross": r["eq_gross"][i],
                "eq_net": r["eq_net"][i],
                "rp_gross": r["rp_gross"][i],
                "rp_net": r["rp_net"][i],
                "turnover": r["turnover"][i] if i < len(r["turnover"]) else np.nan,
                "trading_cost_paid": r["tc_paid"][i] if i < len(r["tc_paid"]) else np.nan,
                "cost_coeff": cost_series_global[i + 1] if i + 1 < len(cost_series_global) else np.nan,
            })

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"timeseries_seed_{seed}.csv"), index=False)
    return df


def save_weights_csv(seed_dir, seed, times, chosen_assets, results):
    rows = []
    date_index = pd.to_datetime(times)

    for r in results:
        W = r["W"]
        W_star = r["W_star"]

        for t, dt in enumerate(date_index):
            row = {
                "seed": seed,
                "date": dt,
                "strategy": r["name"],
                "weight_type": "executed"
            }
            for j, asset in enumerate(chosen_assets):
                row[str(asset)] = W[t, j]
            rows.append(row)

            if not np.isnan(W_star).all():
                row_star = {
                    "seed": seed,
                    "date": dt,
                    "strategy": r["name"],
                    "weight_type": "target"
                }
                for j, asset in enumerate(chosen_assets):
                    row_star[str(asset)] = W_star[t, j]
                rows.append(row_star)

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"weights_seed_{seed}.csv"), index=False)
    return df


def save_cost_series_csv(seed_dir, seed, times, cost_series):
    df = pd.DataFrame({
        "seed": seed,
        "date": pd.to_datetime(times),
        "cost_coeff": cost_series
    })
    df.to_csv(os.path.join(seed_dir, f"cost_series_seed_{seed}.csv"), index=False)
    return df


def save_sanity_csv(seed_dir, seed, results):
    rows = []
    for r in results:
        rows.append({
            "seed": seed,
            "strategy": r["name"],
            "has_nan_weights": bool(np.isnan(r["W"]).any()),
            "avg_max_weight": float(np.mean(np.max(r["W"], axis=1))),
            "avg_turnover": float(np.mean(turnover_from_weights(r["W"]))) if r["W"].shape[0] > 1 else np.nan,
            "max_turnover": float(np.max(turnover_from_weights(r["W"]))) if r["W"].shape[0] > 1 else np.nan,
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"sanity_seed_{seed}.csv"), index=False)
    return df


def save_failed_seed_csv(seed_dir, seed, error_message):
    df = pd.DataFrame([{
        "seed": seed,
        "error": str(error_message)
    }])
    df.to_csv(os.path.join(seed_dir, f"failed_seed_{seed}.csv"), index=False)
    return df


# ============================================================
# 15) Plot helpers
# ============================================================
def plot_grouped_results(times, results, cost_series):
    date_full = pd.to_datetime(times)
    date_idx = pd.to_datetime(times[1:])

    plt.figure(figsize=(10, 5))
    plt.plot(date_full, cost_series)
    plt.title("Time-varying transaction cost coefficient")
    plt.xlabel("Date")
    plt.ylabel("Cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    for r in results:
        plt.plot(date_idx, r["eq_net"], label=r["name"])
    plt.title("Net equity curves")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    for r in results:
        plt.plot(date_idx, r["turnover"], label=r["name"])
    plt.title("Turnover through time")
    plt.xlabel("Date")
    plt.ylabel("Turnover")
    plt.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    for r in results:
        plt.plot(date_idx, r["tc_paid"], label=r["name"])
    plt.title("Realised trading costs through time")
    plt.xlabel("Date")
    plt.ylabel("Trading cost")
    plt.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()


# ============================================================
# 16) Single-seed run with saving
# ============================================================
def run_single_seed_all_comparisons(
    seed=7,
    data=None,
    out_dir="all_combined",
    n_assets=50,
    lookback_sigma=252,
    K_simple=60,
    ar_window=252,
    smooth_alpha=0.20,
    ewma_alpha=0.90,
    sample_shrink_delta=0.10,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-6,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha_cost=2.0,
    annualize=252,
    save_weights=True,
    make_plots=False
):
    if data is None:
        data = qndata.stocks.load_data()

    seed_dir = os.path.join(out_dir, f"seed_{seed}")
    ensure_dir(seed_dir)

    data, chosen_assets = load_quantiacs_universe(
        seed=seed,
        n_assets=n_assets,
        data=data
    )

    print(f"\nRunning seed {seed} | first 10 assets: {chosen_assets[:10]}")

    save_chosen_assets_csv(seed_dir, seed, chosen_assets)

    times, R_daily = compute_returns_matrix(
        data,
        chosen_assets,
        use_log_returns=use_log_returns
    )

    Sigma_sample = compute_sigma_series_sample(
        R_daily,
        lookback=lookback_sigma,
        annualize=annualize,
        shrink_delta=sample_shrink_delta,
        ridge=ridge
    )

    Sigma_ewma = compute_sigma_series_ewma(
        R_daily,
        lookback_init=lookback_sigma,
        annualize=annualize,
        alpha=ewma_alpha,
        ridge=ridge
    )

    mu_simple = mu_series_simple(
        R_daily,
        lookback_mu=K_simple,
        mode="momentum",
        annualize=annualize
    )

    mu_ar1 = mu_series_ar1(
        R_daily,
        lookback_model=ar_window,
        annualize=annualize,
        smooth_alpha=smooth_alpha
    )

    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha_cost
    )

    global cost_series_global
    cost_series_global = cost_series

    combo_specs = [
        ("Simple alpha + SampleCov + Basic + One-Step", mu_simple, Sigma_sample, "basic", "one"),
        ("Simple alpha + SampleCov + Basic + Two-Step", mu_simple, Sigma_sample, "basic", "two"),
        ("Simple alpha + SampleCov + Enhanced + One-Step", mu_simple, Sigma_sample, "enhanced", "one"),
        ("Simple alpha + SampleCov + Enhanced + Two-Step", mu_simple, Sigma_sample, "enhanced", "two"),

        ("Simple alpha + EWMA + Basic + One-Step", mu_simple, Sigma_ewma, "basic", "one"),
        ("Simple alpha + EWMA + Basic + Two-Step", mu_simple, Sigma_ewma, "basic", "two"),
        ("Simple alpha + EWMA + Enhanced + One-Step", mu_simple, Sigma_ewma, "enhanced", "one"),
        ("Simple alpha + EWMA + Enhanced + Two-Step", mu_simple, Sigma_ewma, "enhanced", "two"),

        ("AR1 alpha + SampleCov + Basic + One-Step", mu_ar1, Sigma_sample, "basic", "one"),
        ("AR1 alpha + SampleCov + Basic + Two-Step", mu_ar1, Sigma_sample, "basic", "two"),
        ("AR1 alpha + SampleCov + Enhanced + One-Step", mu_ar1, Sigma_sample, "enhanced", "one"),
        ("AR1 alpha + SampleCov + Enhanced + Two-Step", mu_ar1, Sigma_sample, "enhanced", "two"),

        ("AR1 alpha + EWMA + Basic + One-Step", mu_ar1, Sigma_ewma, "basic", "one"),
        ("AR1 alpha + EWMA + Basic + Two-Step", mu_ar1, Sigma_ewma, "basic", "two"),
        ("AR1 alpha + EWMA + Enhanced + One-Step", mu_ar1, Sigma_ewma, "enhanced", "one"),
        ("AR1 alpha + EWMA + Enhanced + Two-Step", mu_ar1, Sigma_ewma, "enhanced", "two"),
    ]

    results = []

    for name, mu_series, Sigma_series, family, method in combo_specs:
        print(f"Running {name} ...")

        W, W_star = run_strategy_combo(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            family=family,
            method=method,
            gamma=gamma,
            eta=eta,
            rho=rho,
            kappa=kappa,
            tau=tau,
            w_max=w_max,
            ridge=ridge
        )

        res = summarise_strategy(
            name=name,
            W=W,
            W_star=W_star,
            R_daily=R_daily,
            cost_series=cost_series
        )
        results.append(res)

    summary_df = save_summary_csv(seed_dir, seed, results)
    timeseries_df = save_timeseries_csv(seed_dir, seed, times, results)
    save_cost_series_csv(seed_dir, seed, times, cost_series)
    save_sanity_csv(seed_dir, seed, results)

    if save_weights:
        save_weights_csv(seed_dir, seed, times, chosen_assets, results)

    if make_plots:
        plot_grouped_results(times, results, cost_series)

    return {
        "seed": seed,
        "chosen_assets": chosen_assets,
        "times": times,
        "R_daily": R_daily,
        "Sigma_sample": Sigma_sample,
        "Sigma_ewma": Sigma_ewma,
        "mu_simple": mu_simple,
        "mu_ar1": mu_ar1,
        "cost_series": cost_series,
        "results": results,
        "summary_df": summary_df,
        "timeseries_df": timeseries_df,
    }


# ============================================================
# 17) Batch run across 25 seeds + combined CSVs
# ============================================================
def run_many_seeds_all_comparisons(
    seeds=range(1, 26),
    out_dir="all_combined",
    n_assets=50,
    lookback_sigma=252,
    K_simple=60,
    ar_window=252,
    smooth_alpha=0.20,
    ewma_alpha=0.90,
    sample_shrink_delta=0.10,
    use_log_returns=False,
    gamma=5.0,
    eta=2.0,
    rho=25.0,
    kappa=0.10,
    tau=0.05,
    w_max=0.10,
    ridge=1e-6,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha_cost=2.0,
    annualize=252,
    save_weights=True,
    make_plots=False
):
    ensure_dir(out_dir)

    print("Loading Quantiacs data once...")
    data = qndata.stocks.load_data()

    all_summary = []
    all_timeseries = []
    failed_rows = []

    for seed in list(seeds):
        try:
            out = run_single_seed_all_comparisons(
                seed=seed,
                data=data,
                out_dir=out_dir,
                n_assets=n_assets,
                lookback_sigma=lookback_sigma,
                K_simple=K_simple,
                ar_window=ar_window,
                smooth_alpha=smooth_alpha,
                ewma_alpha=ewma_alpha,
                sample_shrink_delta=sample_shrink_delta,
                use_log_returns=use_log_returns,
                gamma=gamma,
                eta=eta,
                rho=rho,
                kappa=kappa,
                tau=tau,
                w_max=w_max,
                ridge=ridge,
                base_cost=base_cost,
                cost_mode=cost_mode,
                vol_lookback=vol_lookback,
                alpha_cost=alpha_cost,
                annualize=annualize,
                save_weights=save_weights,
                make_plots=make_plots
            )
            all_summary.append(out["summary_df"])
            all_timeseries.append(out["timeseries_df"])

        except Exception as e:
            print(f"Seed {seed} failed: {e}")
            seed_dir = os.path.join(out_dir, f"seed_{seed}")
            ensure_dir(seed_dir)
            save_failed_seed_csv(seed_dir, seed, e)
            failed_rows.append({
                "seed": seed,
                "error": str(e)
            })

    combined_summary = None
    combined_timeseries = None
    avg_by_strategy = None
    avg_timeseries = None

    if len(all_summary) > 0:
        combined_summary = pd.concat(all_summary, ignore_index=True)
        combined_summary.to_csv(os.path.join(out_dir, "all_seeds_summary.csv"), index=False)

        avg_by_strategy = (
            combined_summary
            .groupby("strategy", as_index=False)
            .agg({
                "final_equity_gross": "mean",
                "final_equity_net": "mean",
                "cum_return_gross": "mean",
                "cum_return_net": "mean",
                "sharpe_gross": "mean",
                "sharpe_net": "mean",
                "max_drawdown_gross": "mean",
                "max_drawdown_net": "mean",
                "avg_turnover": "mean",
                "max_turnover": "mean",
                "avg_max_weight": "mean",
                "avg_cost_coeff": "mean",
                "avg_realised_tc": "mean",
            })
            .sort_values(["sharpe_net", "final_equity_net"], ascending=[False, False])
            .reset_index(drop=True)
        )
        avg_by_strategy.to_csv(os.path.join(out_dir, "average_metrics_across_seeds.csv"), index=False)

        print("\n=== Average metrics across seeds ===")
        print(avg_by_strategy)

    if len(all_timeseries) > 0:
        combined_timeseries = pd.concat(all_timeseries, ignore_index=True)
        combined_timeseries.to_csv(os.path.join(out_dir, "all_seeds_timeseries.csv"), index=False)

        avg_timeseries = (
            combined_timeseries
            .groupby(["date", "strategy"], as_index=False)
            .agg({
                "eq_gross": "mean",
                "eq_net": "mean",
                "rp_gross": "mean",
                "rp_net": "mean",
                "turnover": "mean",
                "trading_cost_paid": "mean",
                "cost_coeff": "mean",
            })
        )
        avg_timeseries.to_csv(os.path.join(out_dir, "average_timeseries_across_seeds.csv"), index=False)

    if len(failed_rows) > 0:
        pd.DataFrame(failed_rows).to_csv(os.path.join(out_dir, "failed_seeds.csv"), index=False)

    return {
        "combined_summary": combined_summary,
        "combined_timeseries": combined_timeseries,
        "avg_by_strategy": avg_by_strategy,
        "avg_timeseries": avg_timeseries,
    }


# ============================================================
# 18) Main
# ============================================================
if __name__ == "__main__":
    out = run_many_seeds_all_comparisons(
        seeds=range(1, 26),
        out_dir="all_combined",
        n_assets=50,
        lookback_sigma=252,
        K_simple=60,
        ar_window=252,
        smooth_alpha=0.20,
        ewma_alpha=0.90,
        sample_shrink_delta=0.10,
        use_log_returns=False,
        gamma=5.0,
        eta=2.0,
        rho=25.0,
        kappa=0.10,
        tau=0.05,
        w_max=0.10,
        ridge=1e-6,
        base_cost=0.0020,
        cost_mode="vol_scaled",
        vol_lookback=21,
        alpha_cost=2.0,
        annualize=252,
        save_weights=True,    # set False if files get too large
        make_plots=False
    )

    print("\nAll done. Results saved in folder: all_combined")